# Trích xuất keyframe (vlm_keyframes): OmniShotCut → quality gate → PE-Core-L → clustering → shot MMR

Kaggle: bật **Internet + T4 GPU**. Dùng Python và PyTorch/torchvision có sẵn của Kaggle;
cài dependencies bổ sung trực tiếp bằng pip, không tạo venv. Chạy 5 cell code theo thứ tự.

- NVDEC đọc video vào GPU, bộ đệm hữu hạn; preprocess/model/clustering/histogram trên GPU.
- OmniShotCut dùng model, quy tắc overlap/padding/query/stitch của upstream v1.5.
  Chỉ thay nguồn frame và validation transform để giữ tensor trên GPU. Không lưu RGB toàn video.
- OmniShotCut xét toàn video. Shot ngắn dưới 1 giây được gộp với shot kế bên cho
  đến khi không còn shot dưới 1 giây (trừ video chỉ có một shot rất ngắn). Một lượt
  NVDEC chung kiểm tra chất lượng mẫu 3 FPS và đưa mẫu đạt sang PE-Core-L14-336.
- Gate không xét tỷ lệ pixel tối; số bin màu của toàn khung hình <=5 coi là gần đồng màu. Nếu toàn bộ mẫu
  của shot đồng màu thì bỏ shot; nếu chỉ trượt blur/cháy sáng/tương phản, giữ mẫu
  không đồng màu có điểm cao nhất.
- PKL chứa float32 features + sharpness + frame_indexes/timestamps gốc + sample_ranges/sample_fps;
  CSV map giữ nguyên cột và chỉ số/thời gian gốc (kể cả VFR), JPEG giữ độ phân giải gốc.
- Clustering theo chunk 20 giây tạo candidate, nhưng common anchor và MMR dùng shot Omni gốc.
- Anchor và MMR toàn cục cùng điểm 0.4 đại diện - 0.6 giống ảnh đã chọn.
- Sau MMR, so mọi cặp cosine >0.97. Với mỗi cặp trùng, giữ ảnh có độ khác biệt
  với các keyframe lân cận và độ nét tốt hơn (60%/40%). Điểm nét được lưu từ
  quality gate; không decode/embed lại. Shot có thể mất ảnh nếu trùng shot khác.
- Xuất JPEG đọc video lại bằng NVDEC, encode CUDA/nvJPEG; CPU chỉ ghi byte JPEG, ZIP STORE và rclone.
- Tải một video → xử lý → ZIP ảnh + scene TXT + embedding PKL + map CSV → upload → xóa job.
  Lỗi thì giữ job và dừng. Không xuất shot MP4. Không tái sử dụng workspace SurgVLP cũ.

Notebook dùng GPU **0**; hai T4 không tự gộp VRAM. NVDEC không hỗ trợ runtime/codec sẽ báo lỗi,
không âm thầm chuyển sang CPU. GPU resize/color conversion có thể khác pixel nhỏ so với CPU cũ;
không cam kết biên scene/keyframe giống tuyệt đối từng frame. Cần kiểm thử trên T4 thực tế.


In [ ]:
from pathlib import Path
import json, os, re, shutil, subprocess, sys
from IPython.display import display

WORK = Path('/kaggle/working/vlm_pipeline')
WORK.mkdir(parents=True,exist_ok=True)
PYTHON = sys.executable
JOBS_ROOT = WORK/'jobs'
START_VIDEO_ID, END_VIDEO_ID = 1, 1  # Default: verify LHE01; change this range as needed.
DOWNLOAD_ROOT_FOLDER_ID = '1yPb3EQBPKfKtlP0PGGZHmKDF3pghnRCN'
UPLOAD_ROOT_FOLDER_ID = '11__IfRj1eLK4GZWEeNLs1SQK-DowEU1K'
RCLONE_CONFIG_SOURCE = Path('/kaggle/input/datasets/vnnoname/rclone-new/rclone.conf')
RCLONE_REMOTE = 'iamtro'
CONFIG = dict(
    pipeline_id='vlm-neighbor-sharpness-v1',
    source_root=str(WORK/'source'), weights_cache=str(WORK/'weights'),
    omni_weights=None, batch_size=16, decode_batch_size=8, sample_fps=3.0,
    max_scene_seconds=20.0, mmr_budget_fraction=0.50, anchors_per_shot=3,
    mmr_relevance_weight=0.40, final_similarity_threshold=0.97,
    min_hist_bins=5, overwrite=False,
)
if 'mmr_max_similarity' in CONFIG or CONFIG['pipeline_id'] != 'vlm-neighbor-sharpness-v1':
    raise RuntimeError('Stale VLM notebook configuration; open the updated vlm_keyframes.ipynb')
print('VLM pipeline:',CONFIG['pipeline_id'], 'range:',START_VIDEO_ID,END_VIDEO_ID,
      'final cosine threshold:',CONFIG['final_similarity_threshold'])

def run(command,env=None):
    command=[str(x) for x in command]
    process=subprocess.Popen(command,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,
                             text=True,env=env,bufsize=1)
    bars={}
    for line in process.stdout:
        if re.search(r'\d+%\|',line):
            stage=line.split(':',1)[0]
            payload={'text/plain':line.rstrip()}
            if stage not in bars: bars[stage]=display(payload,raw=True,display_id=True)
            else: bars[stage].update(payload,raw=True)
        else: print(line,end='',flush=True)
    if process.wait(): raise RuntimeError(f'Command failed ({process.returncode}): {command}')


Cell code 2: cài trực tiếp vào Python của Kaggle. Dùng PyTorch/torchvision sẵn có, không ép Python 3.11 và không tạo venv.

In [ ]:
REQUIREMENTS = "# Keep Kaggle's installed scientific/PyTorch stack when requirements are satisfied.\nnumpy\ntorch\ntorchvision\npandas\nscipy\nscikit-learn\nPillow\ntimm==1.0.15\neinops>=0.8.1\nomegaconf==2.3.0\nhuggingface-hub>=0.28.1\nftfy\nregex\niopath==0.1.10\nPyYAML\ntqdm\nffmpeg-python==0.2.0\nPyNvVideoCodec>=2.0.2,<3\n"
packages=[line.strip() for line in REQUIREMENTS.splitlines() if line.strip() and not line.startswith('#')]
run([PYTHON,'-m','pip','install','--disable-pip-version-check',*packages])
# Kaggle normally includes OpenCV; install only if absent.
import importlib.util
if importlib.util.find_spec('cv2') is None:
    run([PYTHON,'-m','pip','install','opencv-python-headless'])
run([PYTHON,'-m','pip','install','--no-deps','--no-build-isolation',
     'git+https://github.com/facebookresearch/perception_models.git@3e352cca660658d4b5c90f42a7808b11469e4c66',
     'git+https://github.com/UVA-Computer-Vision-Lab/OmniShotCut.git@40ecf12a0f6ace617680cf22b74da0bef4af3c21'])


Cell code 3: rclone và kiểm tra đủ video trong range trước khi tải. Config được copy để refresh token.

In [ ]:
"""Kaggle rclone installer reused from the embedding notebook; needs WORK and run."""
import os
import shutil
import subprocess
import urllib.request
import zipfile

# downloads.rclone.org có thể trả 403 trên Kaggle. Ưu tiên binary sẵn có,
# sau đó apt của hệ thống; nếu apt không dùng được thì thử release GitHub.
rclone = shutil.which("rclone")
if rclone is None and shutil.which("apt-get") and os.geteuid() == 0:
    apt = shutil.which("apt-get")
    apt_env = {**os.environ, "DEBIAN_FRONTEND": "noninteractive"}
    try:
        install = subprocess.run([apt, "install", "-y", "-qq", "rclone"], env=apt_env,
                                 capture_output=True, text=True, timeout=300)
        if install.returncode != 0:
            subprocess.run([apt, "update", "-qq"], env=apt_env, check=True,
                           capture_output=True, text=True, timeout=300)
            run([apt, "install", "-y", "-qq", "rclone"], env=apt_env)
        rclone = shutil.which("rclone")
    except (subprocess.CalledProcessError, subprocess.TimeoutExpired) as exc:
        print("apt không cài được rclone:", exc)

if rclone is None:
    release = "v1.75.2"
    archive = f"rclone-{release}-linux-amd64.zip"
    url = f"https://github.com/rclone/rclone/releases/download/{release}/{archive}"
    rclone_zip = WORK / archive
    try:
        request = urllib.request.Request(url, headers={"User-Agent": "kaggle-rclone-setup"})
        with urllib.request.urlopen(request, timeout=120) as src, rclone_zip.open("wb") as dst:
            shutil.copyfileobj(src, dst)
        with zipfile.ZipFile(rclone_zip) as zf:
            matches = [item for item in zf.infolist() if item.filename.endswith("/rclone") and not item.is_dir()]
            if len(matches) != 1:
                raise RuntimeError("Release rclone không chứa đúng một binary")
            binary = WORK / "rclone"
            with zf.open(matches[0]) as src, binary.open("wb") as dst:
                shutil.copyfileobj(src, dst)
        binary.chmod(0o755)
        rclone = str(binary)
    except Exception as exc:
        raise RuntimeError("Không cài được rclone qua apt hoặc GitHub release. "
                           "Hãy gắn rclone binary vào Kaggle và thêm nó vào PATH.") from exc
run([rclone, "version"])

BATCH_SOURCE = '"""Drive range/download/ZIP/upload/cleanup for the standalone VLM notebook."""\nimport csv\nfrom pathlib import Path\nimport shutil\nimport zipfile\n\n\ndef range_names(start_id, end_id):\n    if (type(start_id) is not int or type(end_id) is not int\n            or not 1 <= start_id <= end_id <= 75):\n        raise ValueError(\'Range must satisfy 1 <= START_VIDEO_ID <= END_VIDEO_ID <= 75\')\n    return [f\'LHE{index:02d}.mp4\' for index in range(start_id, end_id + 1)]\n\n\ndef select_remote_jobs(entries, start_id, end_id):\n    """Require every requested file exactly once before starting any downloads."""\n    names = range_names(start_id, end_id)\n    selected = {name: [] for name in names}\n    for entry in entries:\n        name = entry.get(\'Name\')\n        if not entry.get(\'IsDir\', False) and name in selected:\n            if entry.get(\'Path\', name) != name:\n                raise ValueError(f\'Expected source files directly inside the Drive root: {name}\')\n            selected[name].append(entry)\n    jobs = []\n    for name in names:\n        matches = selected[name]\n        if len(matches) != 1:\n            raise ValueError(f\'Expected exactly one {name} in source Drive folder; found {len(matches)}\')\n        size = matches[0].get(\'Size\', 0)\n        if type(size) is not int or size <= 0:\n            raise ValueError(f\'Invalid source video size: {name}\')\n        jobs.append(dict(source_name=name, video_id=Path(name).stem, group=\'LHE\', source_bytes=size))\n    return jobs\n\n\ndef cleanup_video(work_dir, jobs_root):\n    """Delete only the isolated per-video workspace, never input/Drive/shared runtime."""\n    work_dir, jobs_root = Path(work_dir), Path(jobs_root).resolve()\n    resolved = work_dir.resolve()\n    allowed_ids = {Path(name).stem for name in range_names(1, 75)}\n    if (work_dir.is_symlink() or resolved.parent != jobs_root or resolved.name not in allowed_ids):\n        raise ValueError(f\'Refusing cleanup outside the per-video workspace: {resolved}\')\n    shutil.rmtree(resolved)\n\n\ndef process_remote_video(job, jobs_root, config, rclone, config_path, remote,\n                         download_root_id, upload_root_id, run, extract):\n    """Download one video, extract/upload four outputs, then remove its entire workspace."""\n    allowed_names = set(range_names(1, 75))\n    if (job[\'source_name\'] not in allowed_names or job[\'video_id\'] != Path(job[\'source_name\']).stem\n            or job[\'group\'] != \'LHE\' or not download_root_id or not upload_root_id\n            or download_root_id == upload_root_id):\n        raise ValueError(\'Invalid source job or Drive folder configuration\')\n    remote = remote.rstrip(\':\')\n    if not remote or \':\' in remote:\n        raise ValueError(\'Invalid rclone remote\')\n    jobs_root = Path(jobs_root).resolve()\n    jobs_root.mkdir(parents=True, exist_ok=True)\n    work = jobs_root/job[\'video_id\']\n    if work.is_symlink() or work.resolve().parent != jobs_root:\n        raise ValueError(\'Video workspace must be inside jobs_root\')\n    work.mkdir(exist_ok=True)\n    video = work/job[\'source_name\']\n    # rclone checks transfers and raises through run() on failure; existing downloads can resume.\n    run([rclone, \'--config\', config_path, \'--drive-root-folder-id\', download_root_id,\n         \'--checksum\', \'--retries\', \'3\', \'copyto\', f"{remote}:{job[\'source_name\']}", video])\n    if not video.is_file() or video.stat().st_size != job[\'source_bytes\']:\n        raise ValueError(f\'Download missing or wrong size: {video}\')\n    output = work/\'output\'\n    video_config = dict(config, video=str(video), group=job[\'group\'], output_dir=str(output))\n    extract(video_config, work/\'config.json\')\n    archive, mapping, scenes, embeddings, count = package_keyframes(output, work/\'archives\', job)\n    destinations = upload_keyframes(rclone, config_path, remote, upload_root_id,\n                                    archive, mapping, scenes, embeddings, job, run)\n    # This line is reached only after all four upload commands exit successfully.\n    cleanup_video(work, jobs_root)\n    return dict(video=job[\'source_name\'], keyframes=count, destinations=destinations)\n\n\ndef package_keyframes(output_dir, archive_dir, job):\n    """Flat JPG ZIP per video plus a separate canonical map; reject partial outputs."""\n    root = Path(output_dir)\n    group, video_id = Path(job[\'group\']), job[\'video_id\']\n    image_dir = root/\'keyframes\'/group/video_id\n    mapping = root/\'map_keyframes\'/group/f\'{video_id}.csv\'\n    with mapping.open(newline=\'\', encoding=\'utf-8\') as handle:\n        reader = csv.DictReader(handle)\n        required = {\'keyframe_id\', \'video_id\', \'frame_idx\', \'timestamp_sec\', \'fps\'}\n        if not required.issubset(reader.fieldnames or []):\n            raise ValueError(f\'Invalid keyframe map columns: {mapping}\')\n        rows = list(reader)\n    if not rows:\n        raise ValueError(f\'No keyframes selected: {mapping}\')\n    expected = [f\'{index:06d}.jpg\' for index in range(len(rows))]\n    if ([int(row[\'keyframe_id\']) for row in rows] != list(range(len(rows)))\n            or any(row[\'video_id\'] != video_id for row in rows)):\n        raise ValueError(f\'Keyframe IDs/video IDs do not match: {mapping}\')\n    frames = [int(row[\'frame_idx\']) for row in rows]\n    if frames != sorted(set(frames)) or frames[0] < 0:\n        raise ValueError(f\'Invalid source frame indexes: {mapping}\')\n    actual = sorted(path.name for path in image_dir.iterdir() if path.is_file())\n    if actual != expected or any((image_dir/name).stat().st_size == 0 for name in expected):\n        raise ValueError(f\'Images do not match keyframe map: {image_dir}\')\n    archive = Path(archive_dir)/\'keyframes\'/group/f\'{video_id}.zip\'\n    archive.parent.mkdir(parents=True, exist_ok=True)\n    temporary = archive.with_suffix(\'.zip.partial\')\n    try:\n        # JPEG is already compressed. STORE avoids spending CPU recompressing it.\n        with zipfile.ZipFile(temporary, \'w\', compression=zipfile.ZIP_STORED, allowZip64=True) as handle:\n            for name in expected:\n                handle.write(image_dir/name, arcname=name)\n        with zipfile.ZipFile(temporary) as handle:\n            if handle.namelist() != expected or handle.testzip() is not None:\n                raise ValueError(f\'Invalid keyframe ZIP: {temporary}\')\n        temporary.replace(archive)\n    finally:\n        temporary.unlink(missing_ok=True)\n    scenes = root/\'scenes\'/group/f\'{video_id}.scenes.txt\'\n    embeddings = root/\'embeddings\'/group/f\'{video_id}.pkl\'\n    for path in [scenes, embeddings]:\n        if not path.is_file() or path.stat().st_size == 0:\n            raise ValueError(f\'Missing pipeline cache: {path}\')\n    return archive, mapping, scenes, embeddings, len(rows)\n\n\ndef upload_keyframes(rclone, config_path, remote, root_folder_id, archive, mapping, scenes, embeddings, job, run):\n    """Upload the original four outputs, publishing the completion map last."""\n    remote = remote.rstrip(\':\')\n    if not remote or \':\' in remote or not root_folder_id:\n        raise ValueError(\'Set a valid rclone remote and Drive root_folder_id\')\n    common = [rclone, \'--config\', config_path, \'--drive-root-folder-id\', root_folder_id,\n              \'--checksum\', \'--retries\', \'3\']\n    group = Path(job[\'group\']).as_posix()\n    video_id = job[\'video_id\']\n    destinations = [f\'{remote}:keyframes/{group}/{video_id}.zip\',\n                    f\'{remote}:scenes/{group}/{video_id}.scenes.txt\',\n                    f\'{remote}:embeddings/{group}/{video_id}.pkl\',\n                    f\'{remote}:map_keyframes/{group}/{video_id}.csv\']\n    # Publish the map only after frames and both caches uploaded successfully.\n    for source, destination in zip([archive, scenes, embeddings, mapping], destinations):\n        run([*common, \'copyto\', source, destination])\n    return destinations\n'
exec(BATCH_SOURCE)
RCLONE_CONFIG=WORK/'rclone.conf'
if RCLONE_CONFIG_SOURCE.is_file():
    shutil.copyfile(RCLONE_CONFIG_SOURCE,RCLONE_CONFIG)
else:
    secret=os.environ.get('RCLONE_CONFIG')
    if not secret:
        from kaggle_secrets import UserSecretsClient
        secret=UserSecretsClient().get_secret('RCLONE_CONFIG')
    RCLONE_CONFIG.write_text(secret,encoding='utf-8')
    del secret
RCLONE_CONFIG.chmod(0o600)
if DOWNLOAD_ROOT_FOLDER_ID==UPLOAD_ROOT_FOLDER_ID:
    raise ValueError('Source and output Drive folders must differ')
entries=json.loads(subprocess.check_output([
    str(rclone),'--config',str(RCLONE_CONFIG),'--drive-root-folder-id',DOWNLOAD_ROOT_FOLDER_ID,
    'lsjson',RCLONE_REMOTE.rstrip(':')+':','--files-only','--max-depth','1'],text=True))
jobs=select_remote_jobs(entries,START_VIDEO_ID,END_VIDEO_ID)
print(f'Selected {len(jobs)} videos: {jobs[0]["source_name"]} ... {jobs[-1]["source_name"]}')


Cell code 4: cài source chính được nhúng trong notebook và kiểm tra CUDA/NVDEC imports. Không cần clone repo riêng.

In [ ]:
SOURCE_ZIP = '...'
import base64, io, zipfile
SOURCE_ROOT=WORK/'source'
SOURCE_ROOT.mkdir(exist_ok=True)
with zipfile.ZipFile(io.BytesIO(base64.b64decode(SOURCE_ZIP))) as bundle:
    for name in bundle.namelist():
        target=(SOURCE_ROOT/name).resolve()
        if not target.is_relative_to(SOURCE_ROOT.resolve()): raise ValueError('Invalid bundled path')
        target.parent.mkdir(parents=True,exist_ok=True)
        target.write_bytes(bundle.read(name))
smoke=WORK/'smoke.py'
smoke.write_text("""
import sys, shutil, torch, torchvision, PyNvVideoCodec as nvc, omnishotcut
assert shutil.which('ffprobe'), 'ffprobe is required to read source timestamps'
assert torch.cuda.is_available(), 'Select T4 GPU'
torch.cuda.set_device(0)
assert torch.cuda.get_device_capability(0)[0]>=7, 'Use T4 or newer; P100 is unsupported'
import core.vision_encoder.pe as pe
assert 'PE-Core-L14-336' in pe.CLIP.available_configs()
print('Python:',sys.version.split()[0], 'GPU:',torch.cuda.get_device_name(0))
print('PyTorch:',torch.__version__,'PyNvVideoCodec:',nvc.__version__)
print('Imports OK; actual NVDEC is validated when opening the first video.')
""",encoding='utf-8')
run([PYTHON,smoke])


Cell code 5: chạy từng video. Re-run cell này để tiếp tục sau lỗi; map hoàn tất/scene/embedding được dùng lại với overwrite=False. Khi cập nhật code, chạy cell 4 trước cell 5.

In [ ]:
worker_path=SOURCE_ROOT/'vlm_worker.py'
def extract_video(config,config_path):
    config_path.write_text(json.dumps(config,indent=2),encoding='utf-8')
    run([PYTHON,worker_path,'--config',config_path])

UPLOAD_DONE=globals().get('UPLOAD_DONE',set())
pending=[job for job in jobs if job['video_id'] not in UPLOAD_DONE]
for index,job in enumerate(pending,1):
    print(f'[{index}/{len(pending)}] {job["source_name"]}',flush=True)
    try:
        result=process_remote_video(job,JOBS_ROOT,CONFIG,rclone,RCLONE_CONFIG,RCLONE_REMOTE,
                                    DOWNLOAD_ROOT_FOLDER_ID,UPLOAD_ROOT_FOLDER_ID,run,extract_video)
        UPLOAD_DONE.add(job['video_id'])
        print(f'Uploaded {job["video_id"]}: {result["keyframes"]} images; local job removed.',flush=True)
    except Exception:
        print(f'Failed: {job["source_name"]}; retained: {JOBS_ROOT/job["video_id"]}',flush=True)
        raise
print(f'Completed {len(UPLOAD_DONE.intersection(job["video_id"] for job in jobs))}/{len(jobs)}')
